# TDT13 IMDb sentiment: FINAL TEST RUNS (freezes the configuration)

> **This notebook freezes the configuration and uses the test split.**
> After `freeze`, nothing in the configuration, data, demonstrations or TF-IDF models may
> change: every test run checks `frozen.lock` and refuses to load the test split if anything
> differs. Do not edit `configs/experiment.yaml`, re-run `prepare`/`select-demos`, or refit
> the baselines after this point. Re-freezing after seeing test results is not allowed.

Order of steps:

1. **L4 GPU required** (Runtime → Change runtime type → L4 GPU, Colab Pro). The notebook stops
   on any other GPU, because all test conditions must run on the same GPU type.
2. Before freeze: `git pull`, stop if the git tree is not clean, check the data and
   demonstration fingerprints and the TF-IDF models.
3. Before freeze: run the six secondary `_flip` conditions on `validation-dev` (they need
   development runs before the configuration is frozen) and evaluate `validation-dev`.
   Then check that all 13 LLM conditions have complete dev runs with the current settings.
4. `freeze`, only if `frozen.lock` does not exist yet; otherwise the existing lock is checked
   (and the code is not pulled again).
5. Test: baselines, all 13 LLM conditions (about 5 h on an L4 by the dev pilot's estimate of
   ~2.7 h for 7 conditions), evaluation.

**Resuming:** the repository and all outputs are on Google Drive (`MyDrive/tdt13`). Every LLM
batch is appended and fsync'ed, so after a disconnect, reconnect and choose **Run all**:
finished reviews are skipped and `freeze` is not repeated.

Run `notebooks/colab_gpu_runs.ipynb` first (development runs on `validation-dev`).

## Expected fingerprints

In [ ]:
EXPECTED_SPLIT_FILES = {
    "train_fit": "7c9187b7047e6edfbf894920caecddb23962faee95102a0a4d6f5838c3ab735a",
    "validation": "0e51f1741c152da279c75e98354a8b1d41e124fe4f26858d1c88287bad74a914",
    "test": "0c76b0090c3c4d7f1d40955a93b25d05cd2b8efd913af04a674c065017168161",
}
EXPECTED_DEMO_SETS_SHA256 = "026458fefdc6230ff730c2c01bffd302209e63da4117939764d6bd34cba8ea52"
# demo_sets.json before the secondary _flip conditions were appended (original conditions
# identical). A Drive copy with this hash is regenerated once with select-demos (before freeze).
PRE_FLIP_DEMO_SETS_SHA256 = "d883d9ea5a8034e50be44b0e62a0b3754d3c880d4b086d0d0774c3c3adccca76"

REPO_DIR = "/content/drive/MyDrive/tdt13"
FLIP_CONDITIONS = [f"qwen_{k}shot_set{s}_flip" for k in (2, 4) for s in (1, 2, 3)]

## Helpers and GPU check (L4 required)

In [ ]:
import json, os, shlex, subprocess, sys

def run(cmd, cwd=None):
    """Run a shell command, stream its output unbuffered, stop the notebook if it fails."""
    print(f"$ {cmd}", flush=True)
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    p = subprocess.Popen(cmd, shell=True, cwd=cwd, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    if p.wait() != 0:
        raise RuntimeError(f"Command failed (exit {p.returncode}): {cmd}")

def py(src):
    """Run Python source in a fresh interpreter (uses the installed imdbsent package)."""
    run(f"{shlex.quote(sys.executable)} -c {shlex.quote(src)}")

import torch
assert torch.cuda.is_available(), "No CUDA GPU. Use Runtime -> Change runtime type -> L4 GPU."
GPU_NAME = torch.cuda.get_device_name(0)
print("GPU:", GPU_NAME)
if "L4" not in GPU_NAME:
    raise RuntimeError(f"GPU is {GPU_NAME!r}, not an L4. All test conditions must run on the same "
                       "GPU type (L4). Change the runtime type and run all again.")
import transformers
print("torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
print("transformers (preinstalled):", transformers.__version__)

## Mount Drive, update code, require a clean git tree

Before freeze the code is pulled. After freeze it is **not** pulled again; instead the
notebook checks that the checked-out commit is the one recorded in `frozen.lock`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
assert os.path.isdir(os.path.join(REPO_DIR, ".git")), (
    f"{REPO_DIR} is missing. Run notebooks/colab_gpu_runs.ipynb first.")
os.chdir(REPO_DIR)
# Google Drive does not keep Unix file modes; ignore them so they do not count as changes.
run("git config core.fileMode false")

FROZEN = os.path.isfile("frozen.lock")
if not FROZEN:
    run("git pull --ff-only")
status = subprocess.check_output(["git", "status", "--porcelain"], text=True)
if status.strip():
    raise RuntimeError("Git tree is not clean; commit or discard these changes first:\n" + status)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
print("git HEAD:", head, "| tree clean")
if FROZEN:
    lock_commit = json.load(open("frozen.lock"))["git"]["commit"]
    print("frozen.lock exists; frozen at commit", lock_commit)
    if head != lock_commit:
        raise RuntimeError(f"HEAD {head} differs from the frozen commit {lock_commit}. "
                           f"Check out the frozen commit (git checkout {lock_commit}) before test runs.")

## Install and technical tests

In [ ]:
from packaging.version import Version

torch_ok = Version(torch.__version__.split("+")[0]) >= Version("2.2") and torch.version.cuda is not None
llm_reqs = "requirements-llm.txt"
if torch_ok:
    keep = [l for l in open(llm_reqs).read().splitlines() if not l.strip().lower().startswith("torch")]
    llm_reqs = "/content/requirements-llm-no-torch.txt"
    open(llm_reqs, "w").write("\n".join(keep) + "\n")
    print(f"Keeping Colab torch {torch.__version__}")
run(f"pip install -q -r requirements.txt -r {llm_reqs}")
run("pip install -q -e .")
run("python -m pytest -q")

## Data, demonstrations and fingerprints

Data must already be on Drive from the development notebook. Before freeze, a `demo_sets.json`
from before the `_flip` conditions is regenerated once (original conditions unchanged).

In [ ]:
import hashlib

assert os.path.isfile("data/processed/data_manifest.json"), (
    "data/processed is missing on Drive. Run notebooks/colab_gpu_runs.ipynb first.")
demo_path = "data/processed/demo_sets.json"
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
if not FROZEN and os.path.isfile(demo_path) and sha(demo_path) == PRE_FLIP_DEMO_SETS_SHA256:
    print("demo_sets.json predates the _flip conditions; regenerating")
    run("python -m imdbsent select-demos")

manifest = json.load(open("data/processed/data_manifest.json"))
mismatch = []
print(f"{'file':<16}{'found':<66}expected")
for split, exp in EXPECTED_SPLIT_FILES.items():
    got = manifest["split_files"].get(split)
    print(f"{split:<16}{str(got):<66}{exp}")
    if got != exp:
        mismatch.append(split)
demo_sha = sha(demo_path) if os.path.isfile(demo_path) else None
print(f"{'demo_sets.json':<16}{str(demo_sha):<66}{EXPECTED_DEMO_SETS_SHA256}")
if demo_sha != EXPECTED_DEMO_SETS_SHA256:
    mismatch.append("demo_sets.json")
if mismatch:
    raise RuntimeError(f"Fingerprint mismatch for {mismatch}. Do not freeze or run test.")
print("All fingerprints match.")

## Baselines and TF-IDF models (before freeze)

The trained TF-IDF models are part of the frozen state. They must exist and match the current
TF-IDF settings; validation baselines are produced if missing.

In [ ]:
if not FROZEN:
    conds = ["vader", "vader_fulltext", "tfidf_lr", "tfidf_lr_fulltext"]
    missing = [c for c in conds if not os.path.isfile(f"runs/validation/{c}/run_manifest.json")
               or not json.load(open(f"runs/validation/{c}/run_manifest.json")).get("complete")]
    if missing:
        run("python -m imdbsent baselines --split validation --conditions " + " ".join(missing))
    py("""
import joblib, sys
from imdbsent.config import Paths, load_config
from imdbsent.predio import settings_hash
cfg, paths = load_config(), Paths()
want = settings_hash(cfg, 'tfidf_lr')
bad = []
for c in ('tfidf_lr', 'tfidf_lr_fulltext'):
    p = paths.models / f'{c}.joblib'
    if not p.exists():
        bad.append(f'{c}: model missing')
        continue
    info = joblib.load(p)['info']
    print(c, 'selected C =', info['selected_C'], '| settings match:', info['settings_hash'] == want)
    if info['settings_hash'] != want:
        bad.append(f'{c}: trained with different settings')
if bad:
    sys.exit('TF-IDF models not ready: ' + '; '.join(bad) +
             '. Run: python -m imdbsent baselines --split validation --overwrite')
""")
else:
    print("Frozen: models are checked against frozen.lock below.")

## Secondary `_flip` conditions on validation-dev (before freeze)

The six order conditions need development runs before the configuration is frozen. Runs are
resumable; completed ones are skipped. The evaluation shows the primary dev metrics and the
separate order section (`order_effect.csv`). These are **development results**.

In [ ]:
if not FROZEN:
    for c in FLIP_CONDITIONS:
        run(f"python -m imdbsent llm --condition {c} --split-tag validation-dev")
    run("python -m imdbsent evaluate --split-tag validation-dev")
    import pandas as pd
    pd.set_option("display.width", 250)
    display(pd.read_csv("results/validation-dev/order_effect.csv"))
else:
    print("Frozen: skipping development runs.")

## Pre-freeze check: complete dev runs for all 13 LLM conditions

Every LLM condition must have a complete `validation-dev` run made with the current Qwen
settings and demonstrations before freezing.

In [ ]:
if not FROZEN:
    py("""
import json, sys
from imdbsent.config import Paths, load_config
from imdbsent.demos import condition_names, load_demo_sets
from imdbsent.predio import settings_hash
cfg, paths = load_config(), Paths()
want, demos = settings_hash(cfg, 'qwen'), load_demo_sets(paths)['conditions']
bad = []
for c in condition_names(cfg):
    p = paths.run_dir('validation-dev', c) / 'run_manifest.json'
    if not p.exists():
        bad.append(f'{c}: no dev run'); continue
    m = json.loads(p.read_text())
    ok = (m.get('complete'), m.get('settings_hash') == want, m.get('demo_ids') == demos[c]['demo_ids'])
    print(f'{c:<24} complete={ok[0]!s:<6} settings={ok[1]!s:<6} demos={ok[2]!s:<6} gpu={m.get('hardware', {}).get('gpu')}')
    if not all(ok):
        bad.append(c)
if bad:
    sys.exit('Not ready to freeze: ' + ', '.join(bad))
print('All 13 LLM conditions have complete dev runs with the current settings.')
""")

## Freeze (only once)

From here on the configuration is fixed. `freeze` requires pinned revisions and a clean git
tree, and records the config hash, data, demonstrations, TF-IDF models and git commit in
`frozen.lock`. If the lock exists, it is only checked.

In [ ]:
if not os.path.isfile("frozen.lock"):
    run("python -m imdbsent freeze")
    FROZEN = True
else:
    print("frozen.lock exists; not freezing again")
py("""
import json
from imdbsent.config import Paths, load_config
from imdbsent.guard import check_lock
lock = check_lock(load_config(), Paths())
print('frozen.lock matches the current state. Frozen at', lock['frozen_utc'],
      'commit', lock['git']['commit'], '| warnings:', lock.get('warnings'))
""")

## Test split: baselines, LLM conditions, evaluation

Everything below uses the test split and produces **final results**. The LLM step is
resumable: after a disconnect, reconnect and **Run all**.

In [ ]:
run("python -m imdbsent baselines --split test")

In [ ]:
run("python -m imdbsent llm --condition all --split-tag test")

Check that every LLM test run was made on an L4 (same GPU type for all conditions).

In [ ]:
import glob

gpus = {}
for f in sorted(glob.glob("runs/test/qwen_*/run_manifest.json")):
    m = json.load(open(f))
    gpus[m["condition"]] = (m.get("hardware", {}).get("gpu"), m.get("complete"), m.get("sessions"))
for c, (g, done, sessions) in gpus.items():
    print(f"{c:<24} gpu={g}  complete={done}  sessions={len(sessions or [])}")
not_l4 = [c for c, (g, _, _) in gpus.items() if not g or "L4" not in g]
if not_l4:
    raise RuntimeError(f"Runs not (only) on an L4: {not_l4}")
print("All LLM test runs report an L4 GPU (latest session).")

In [ ]:
run("python -m imdbsent evaluate --split-tag test")
import pandas as pd
pd.set_option("display.width", 250)
print(open("results/test/STATUS.txt").read())
display(pd.read_csv("results/test/metrics_summary.csv"))

Secondary, post-hoc order analysis on test (original vs. `_flip`, same set):

In [ ]:
display(pd.read_csv("results/test/order_effect.csv"))